# AWS Pruning + SageMaker Inference Demo

This notebook demonstrates an end-to-end workflow:

1. Set up and validate AWS infrastructure access with the AWS CLI.
2. Run model pruning + artifact registration to S3.
3. Deploy (or update) a SageMaker endpoint.
4. Invoke prompts and read generation metadata.
5. Download token-level logits artifacts and compute token log probabilities.

The default calibration dataset source is `humanevalplus_prompts`, but the workflow is dataset-agnostic through a single `CALIBRATION_SOURCE` variable.

## Prerequisites (what you need installed and configured)

- **AWS CLI** (`aws`): used for identity checks, optional bucket creation, ECR login, and printed helper commands.
- **Docker** (optional): only needed if you build and push the serving image yourself.
- **Python environment**: this repo’s dependencies (`boto3`, `numpy`, `transformers`, PyTorch for pruning script). If imports fail, use the **Optional: install Python dependencies** cell at the top.
- **Configuration**: copy [`template.env`](template.env) to `.env` at the repository root and fill in your account, buckets, and role. The **imports** cell loads `.env` before reading variables (`.env` is gitignored).
- **IAM permissions**: ability to call STS, S3 (read/write for artifact and logits buckets), SageMaker (create/update endpoints), and ECR if you push images. The SageMaker execution role must trust SageMaker and access your S3 buckets.

**How to run:** Start Jupyter from the **repository root** (or ensure `cwd` is the repo root) so paths like `infra/aws/sagemaker/...` resolve correctly.

> **Cost note:** SageMaker endpoints and GPU instances bill while running. Use the cleanup section at the end when you are finished experimenting.

## Optional: install Python dependencies

Run the **next code cell** once if you see `ModuleNotFoundError` for imports used in this notebook (`boto3`, `numpy`, `transformers`).

The cell tries each import and runs `python -m pip install` only for packages that are missing. If your environment blocks pip from the notebook kernel, copy the printed `pip` command into a terminal using the **same** Python interpreter as Jupyter (`sys.executable`).

**Local pruning** (`infra/aws/sagemaker/prune_and_register.py`) additionally needs **PyTorch** and **`datasets`** (and related stack); the install cell prints a suggested command for those too.

In [2]:
# Install notebook Python dependencies only if imports fail (idempotent on re-run).
import importlib
import subprocess
import sys

# (import_name, pip_distribution_name) — used by cells below; prune script needs extras listed after.
_NOTEBOOK_PACKAGES = [
    ("boto3", "boto3"),
    ("numpy", "numpy"),
    ("transformers", "transformers"),
]

_missing = []
for import_name, pip_name in _NOTEBOOK_PACKAGES:
    try:
        importlib.import_module(import_name)
    except ImportError:
        _missing.append(pip_name)

if _missing:
    _pip_cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *_missing]
    print("Installing missing packages:", ", ".join(_missing))
    print("$", " ".join(_pip_cmd))
    subprocess.check_call(_pip_cmd)
    print("Done. Re-run this cell if another package was pulled in transitively.\n")
else:
    print("Notebook imports OK:", ", ".join(p for _, p in _NOTEBOOK_PACKAGES))

# Manual copy-paste fallback (same interpreter as this kernel).
print(
    "\nIf pip cannot run inside the notebook, run in a terminal:\n"
    f"  {sys.executable} -m pip install --upgrade boto3 numpy transformers\n"
)

# Local pruning subprocess loads Hugging Face datasets + PyTorch; install separately if needed.
_prune_extras = ["torch", "datasets"]
try:
    importlib.import_module("torch")
except ImportError:
    print(
        "For prune_and_register.py you also need PyTorch + datasets. Example:\n"
        f"  {sys.executable} -m pip install --upgrade {' '.join(_prune_extras)}\n"
        "Or from repository root (recommended): pip install -e \".[dev]\""
    )
else:
    try:
        importlib.import_module("datasets")
    except ImportError:
        print(
            "PyTorch is installed but `datasets` may be missing for calibration loading:\n"
            f"  {sys.executable} -m pip install --upgrade datasets"
        )

Installing missing packages: transformers
$ /var/home/three-kingdoms/work/pruning-metrics/.venv/bin/python -m pip install --upgrade transformers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 44.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.6/800.6 kB 47.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]
Done. Re-run this cell if another package was pulled in transitively.


If pip cannot run inside the notebook, run in a terminal:
  /var/home/three-kingdoms/work/pruning-metrics/.venv/bin/python -m pip install --upgrade boto3 numpy transformers

For prune_and_register.py you also need PyTorch + datasets. Example:
  /var/home/three-kingdoms/work/pruning-metrics/.venv/bin/python -m pip install --upgrade torch datasets
Or from repository root (recommended): pip install -e ".[dev]"


In [3]:
# -----------------------------------------------------------------------------
# Imports, local .env loading, and small helpers
# -----------------------------------------------------------------------------
# This cell defines utilities only; it does not call AWS or load large models.
# Copy `template.env` to `.env` at the repository root; variables load here before
# later cells read os.environ. Already-set environment variables are not overwritten.
from __future__ import annotations

import io  # In-memory file-like objects for reading NumPy .npz from S3 bytes
import json
import os
import re  # Parse JSON blob printed by prune script stdout
import shlex  # Safely quote shell arguments that may contain spaces
import subprocess
import time  # Polling endpoint status during deploy
from pathlib import Path


def _find_repo_root() -> Path:
    """Return repository root (contains ``pyproject.toml`` and ``infra/``)."""

    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "infra").is_dir():
            return candidate
    return here


def _load_env_file(path: Path) -> int:
    """Load ``KEY=value`` lines from a dotenv file into ``os.environ``.

    Skips comments and blank lines. Does not override keys already in the
    process environment (so shell exports or system env take precedence).

    Parameters
    ----------
    path:
        Path to ``.env`` (typically next to ``template.env`` at repo root).

    Returns
    -------
    int
        Count of keys newly set from the file.
    """

    if not path.is_file():
        return 0
    n_set = 0
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:].strip()
        if "=" not in line:
            continue
        key, _, val = line.partition("=")
        key = key.strip()
        if not key:
            continue
        val = val.strip()
        if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
            val = val[1:-1]
        if key not in os.environ:
            os.environ[key] = val
            n_set += 1
    return n_set


REPO_ROOT = _find_repo_root()
_env_path = REPO_ROOT / ".env"
_n_env = _load_env_file(_env_path)
if _n_env:
    print(
        f"Loaded {_n_env} variable(s) from {_env_path} "
        "(existing environment variables were not overwritten)."
    )
elif _env_path.is_file():
    print(
        f"{_env_path} exists; no new keys applied (all present keys already set in the process env)."
    )
else:
    print(
        f"No {_env_path} found. Copy template.env to .env at the repo root to load settings in one place."
    )

import boto3  # Official AWS SDK for Python (SageMaker, S3, STS-backed flows)
import numpy as np
from transformers import AutoTokenizer  # Decode token IDs when previewing logprobs


def run_cmd(command: str, env: dict[str, str] | None = None) -> str:
    """Run a shell command and return stdout.

    Parameters
    ----------
    command:
        Shell command to execute.
    env:
        Optional environment variables to merge with current process env.

    Returns
    -------
    str
        Captured standard output text.

    Raises
    ------
    RuntimeError
        If the command exits with a non-zero code.
    """

    # Echo the command so notebook readers see exactly what ran (audit trail).
    print(f"$ {command}")
    # Merge optional env so prune/deploy scripts see the same bucket/region as this notebook.
    merged_env = dict(os.environ)
    if env:
        merged_env.update(env)
    # shell=True: command is a full string; cwd=REPO_ROOT keeps infra/ paths valid.
    completed = subprocess.run(
        command,
        shell=True,
        check=False,
        cwd=REPO_ROOT,
        env=merged_env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    # Non-zero exit: surface stderr and stop so the next cell does not run on bad state.
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {command}"
        )
    return completed.stdout


def parse_s3_uri(s3_uri: str) -> tuple[str, str]:
    """Split an S3 URI into (bucket, key).

    Used when downloading logits artifacts from `logits_s3_uri` returned by inference.
    """

    cleaned = s3_uri.replace("s3://", "", 1)
    # Bucket-only URI (unusual): treat key as empty string.
    if "/" not in cleaned:
        return cleaned, ""
    bucket, key = cleaned.split("/", 1)
    return bucket, key


def log_softmax(logits: np.ndarray) -> np.ndarray:
    """Compute row-wise numerically stable log-softmax.

    Each row is one generation step: vocab-sized logits -> vocab-sized log probabilities
    that sum to 1 (in probability space). Subtracting max before exp avoids overflow.
    """

    max_per_row = np.max(logits, axis=1, keepdims=True)
    shifted = logits - max_per_row
    logsumexp = np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))
    return shifted - logsumexp


# Sanity check: confirm where shell subprocesses will run from.
print(f"Repo root: {REPO_ROOT}")

/var/home/three-kingdoms/work/pruning-metrics/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Repo root: /var/home/three-kingdoms/work/pruning-metrics/notebooks


In [4]:
# -----------------------------------------------------------------------------
# Configuration: edit here or set environment variables before starting Jupyter
# -----------------------------------------------------------------------------
# Defaults below are placeholders; replace with your account ID, buckets, and role.
# Using os.environ.get(...) lets CI or shell exports override without editing the notebook.

# --- Region and identity ---
# Region drives boto3 clients and CLI `--region`; keep it consistent everywhere.
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
# 12-digit account ID appears in IAM ARNs and ECR image registry hostnames.
AWS_ACCOUNT_ID = os.environ.get("AWS_ACCOUNT_ID", "123456789012")
# Role SageMaker assumes to access S3 and ECR at runtime (must exist in IAM).
SAGEMAKER_ROLE_ARN = os.environ.get(
    "SAGEMAKER_ROLE_ARN",
    f"arn:aws:iam::{AWS_ACCOUNT_ID}:role/service-role/AmazonSageMaker-ExecutionRole",
)

# --- S3 layout for pruned checkpoints and manifest ---
# Artifact bucket/prefix: written by prune_and_register.py (model shards + manifest JSON).
PRUNING_ARTIFACT_BUCKET = os.environ.get("PRUNING_ARTIFACT_BUCKET", "my-pruning-artifacts")
PRUNING_ARTIFACT_PREFIX = os.environ.get("PRUNING_ARTIFACT_PREFIX", "qwen-pruning")
# Logits bucket/prefix: serving container uploads per-request logits here; can match artifact bucket.
PRUNING_LOGITS_BUCKET = os.environ.get("PRUNING_LOGITS_BUCKET", PRUNING_ARTIFACT_BUCKET)
PRUNING_LOGITS_PREFIX = os.environ.get("PRUNING_LOGITS_PREFIX", "logits")

# --- SageMaker hosting ---
# Endpoint name is stable across updates; deploy script creates new model+config behind it.
PRUNING_ENDPOINT_NAME = os.environ.get("PRUNING_ENDPOINT_NAME", "qwen-pruning-endpoint")
# Instance type must fit your model; larger types cost more per hour.
PRUNING_INSTANCE_TYPE = os.environ.get("PRUNING_INSTANCE_TYPE", "ml.g5.48xlarge")
PRUNING_INSTANCE_COUNT = int(os.environ.get("PRUNING_INSTANCE_COUNT", "1"))

# --- Pruning job (local CPU/GPU: loads BASE_MODEL_ID from Hugging Face or cache) ---
BASE_MODEL_ID = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen2.5-Coder-7B-Instruct")
# Comma-separated percentages; 0 = dense baseline upload, higher = more sparsity.
PRUNING_LEVELS = os.environ.get("PRUNING_LEVELS", "0,20")

# --- Calibration data for WANDA statistics (single switch for dataset choice) ---
# Options understood by infra/aws/sagemaker/calibration_datasets.py:
# - humanevalplus_prompts: HumanEval+ prompts from Hugging Face
# - hf:<dataset_name>:<split>:<text_field>: any HF dataset with a text column
CALIBRATION_SOURCE = os.environ.get(
    "CALIBRATION_SOURCE", "humanevalplus_prompts"
)

# Cap calibration cost: fewer samples/tokens = faster pruning stats collection.
MAX_CALIBRATION_SAMPLES = int(os.environ.get("MAX_CALIBRATION_SAMPLES", "16"))
MAX_CALIBRATION_TOKENS = int(os.environ.get("MAX_CALIBRATION_TOKENS", "512"))

# --- Container image for SageMaker endpoint (must exist in ECR if you deploy) ---
ECR_REPOSITORY_NAME = os.environ.get("ECR_REPOSITORY_NAME", "qwen-serving")
IMAGE_TAG = os.environ.get("IMAGE_TAG", "latest")
# Full URI passed to deploy_endpoint.py --container-image-uri
CONTAINER_IMAGE_URI = (
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
    f"{ECR_REPOSITORY_NAME}:{IMAGE_TAG}"
)

# Snapshot of key settings for quick visual verification in the notebook output.
PARAMETERS = {
    "AWS_REGION": AWS_REGION,
    "AWS_ACCOUNT_ID": AWS_ACCOUNT_ID,
    "SAGEMAKER_ROLE_ARN": SAGEMAKER_ROLE_ARN,
    "PRUNING_ARTIFACT_BUCKET": PRUNING_ARTIFACT_BUCKET,
    "PRUNING_ARTIFACT_PREFIX": PRUNING_ARTIFACT_PREFIX,
    "PRUNING_LOGITS_BUCKET": PRUNING_LOGITS_BUCKET,
    "PRUNING_LOGITS_PREFIX": PRUNING_LOGITS_PREFIX,
    "PRUNING_ENDPOINT_NAME": PRUNING_ENDPOINT_NAME,
    "PRUNING_INSTANCE_TYPE": PRUNING_INSTANCE_TYPE,
    "PRUNING_INSTANCE_COUNT": str(PRUNING_INSTANCE_COUNT),
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "PRUNING_LEVELS": PRUNING_LEVELS,
    "CALIBRATION_SOURCE": CALIBRATION_SOURCE,
}

print(json.dumps(PARAMETERS, indent=2))

{
  "AWS_REGION": "us-east-1",
  "AWS_ACCOUNT_ID": "123456789012",
  "SAGEMAKER_ROLE_ARN": "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole",
  "PRUNING_ARTIFACT_BUCKET": "my-pruning-artifacts",
  "PRUNING_ARTIFACT_PREFIX": "qwen-pruning",
  "PRUNING_LOGITS_BUCKET": "my-pruning-artifacts",
  "PRUNING_LOGITS_PREFIX": "logits",
  "PRUNING_ENDPOINT_NAME": "qwen-pruning-endpoint",
  "PRUNING_INSTANCE_TYPE": "ml.g5.48xlarge",
  "PRUNING_INSTANCE_COUNT": "1",
  "BASE_MODEL_ID": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "PRUNING_LEVELS": "0,20",
  "CALIBRATION_SOURCE": "humanevalplus_prompts"
}


## 1) Validate AWS CLI and identity

**What this section does:** Confirms the AWS CLI is installed, then asks AWS “who am I?” via Security Token Service (STS). That proves your access keys, SSO session, or instance role can call the API.

**What to look for in the output:**

- `UserId`, `Account`, and `Arn` should match the account where you want S3 and SageMaker resources.
- If `get-caller-identity` fails with **NoCredentials**, see **Troubleshooting** in the next section below (`aws configure`, SSO / `aws sso login`, profiles, or env vars).

**Region note:** The notebook also reads `aws configure get region`. If it differs from `AWS_REGION` above, later commands still pass `--region` explicitly where needed, but keeping them aligned avoids confusion.

### Troubleshooting: `NoCredentials` / "Unable to locate credentials"

If `aws sts get-caller-identity` fails with something like:

`An error occurred (NoCredentials): Unable to locate credentials. You can configure credentials by running "aws login".`

**What it means:** The AWS CLI searched its **credential chain** (environment variables, `~/.aws/credentials`, SSO cache, config profiles, and on EC2/SageMaker the instance IAM role) and found nothing usable.

**How to fix (use the path that matches how your organization authenticates):**

1. **IAM Identity Center (SSO)** — Often what CLI v2 means by `aws login`:
   - One-time: `aws configure sso` and follow prompts (start URL, SSO region, account, role). That creates a **named profile**.
   - Whenever the session expires: `aws sso login --profile <your-profile-name>` (some setups expose this as `aws login`; check `aws help login` on your machine).
   - Make Jupyter use that profile: **before** starting the notebook server, run `export AWS_PROFILE=<your-profile-name>` in the same shell, **or** in a notebook cell run `import os; os.environ["AWS_PROFILE"] = "<your-profile-name>"` and then re-run the AWS cells.
   - Optional: set `AWS_REGION` to match the region you use for S3/SageMaker.

2. **Access keys (IAM user or long-lived keys):** Run `aws configure` and enter the access key ID, secret key, and default region. Keys are stored under `~/.aws/credentials` (default profile) or a named profile.

3. **Temporary session keys:** If you use keys from an identity provider or `aws sts assume-role`, set all of `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `AWS_SESSION_TOKEN` in the environment where Jupyter runs.

4. **Terminal works, notebook does not:** The kernel often starts **without** your shell profile. Fix: start Jupyter from a terminal where `aws sts get-caller-identity` already succeeds, or export `AWS_PROFILE` / keys in that same terminal before `jupyter lab`. Alternatively set `os.environ` in the notebook for `AWS_PROFILE` (or keys) and re-run the AWS cells.

5. **Cloud notebook / EC2 with an IAM role:** No access keys are needed if the instance has an **instance profile** (or SageMaker execution role). If you still see `NoCredentials`, confirm the role is attached and that the metadata service is reachable from that environment.

After fixing, verify again with: `aws sts get-caller-identity --output json`

In [5]:
# Block: verify CLI is on PATH and print its version (helps debug "command not found").
run_cmd("aws --version")

# Block: call STS to resolve the current principal; output is JSON we parse for readability.
identity_output = run_cmd("aws sts get-caller-identity --output json")
identity = json.loads(identity_output)
print("Caller identity (who is making AWS API calls):")
print(json.dumps(identity, indent=2))

# Block: optional default region from `aws configure`; may be empty if never set.
configured_region = run_cmd("aws configure get region").strip()
print(f"Configured AWS CLI default region: {configured_region!r}")
if configured_region and configured_region != AWS_REGION:
    print(
        "Warning: notebook AWS_REGION differs from AWS CLI configured region. "
        "Commands below pass --region explicitly where it matters."
    )

$ aws --version
aws-cli/2.34.41 Python/3.14.4 Linux/6.17.7-ba29.fc43.x86_64 exe/x86_64.bazzite.43

$ aws sts get-caller-identity --output json

aws: [ERROR]: An error occurred (NoCredentials): Unable to locate credentials. You can configure credentials by running "aws login".



RuntimeError: Command failed with exit code 253: aws sts get-caller-identity --output json

In [ ]:
# -----------------------------------------------------------------------------
# Optional: ensure S3 buckets exist for artifacts and (optionally) logits
# -----------------------------------------------------------------------------
# Pruning uploads checkpoints; the serving container uploads logits. Both need buckets
# you can write to. Skip this cell if buckets already exist and your IAM can access them.
#
# us-east-1 is special: create-bucket does not use LocationConstraint in some setups.
for bucket_name in sorted({PRUNING_ARTIFACT_BUCKET, PRUNING_LOGITS_BUCKET}):
    # head-bucket succeeds if the bucket exists and you have s3:ListBucket or equivalent.
    check_cmd = f"aws s3api head-bucket --bucket {shlex.quote(bucket_name)}"
    try:
        run_cmd(check_cmd)
        print(f"Bucket exists and is accessible: {bucket_name}")
    except RuntimeError:
        print(f"Creating bucket: {bucket_name}")
        if AWS_REGION == "us-east-1":
            create_cmd = f"aws s3api create-bucket --bucket {shlex.quote(bucket_name)}"
        else:
            create_cmd = (
                "aws s3api create-bucket "
                f"--bucket {shlex.quote(bucket_name)} "
                f"--region {shlex.quote(AWS_REGION)} "
                "--create-bucket-configuration "
                f"LocationConstraint={shlex.quote(AWS_REGION)}"
            )
        run_cmd(create_cmd)

## 2) Configure calibration dataset source

**What this controls:** WANDA-style pruning uses activations on sample text. The string `CALIBRATION_SOURCE` tells `infra/aws/sagemaker/calibration_datasets.py` which texts to load. You do **not** need to edit Python code to swap datasets—only this one variable (or the `CALIBRATION_SOURCE` environment variable).

**Supported patterns:**

- `humanevalplus_prompts` — loads HumanEval+ style prompts (good default for coding models).
- `hf:<dataset_name>:<split>:<text_field>` — generic Hugging Face `datasets.load_dataset` path; the text field must exist on each row.

**Examples:**

- `humanevalplus_prompts` (default)
- `hf:openai_humaneval:test:prompt`
- `hf:mbpp:test:text`

In [ ]:
# Block: show the active source and fail fast if the string is not in the supported set.
# This matches the options implemented in calibration_datasets.py (clear error > silent bug).
print(f"Using calibration source: {CALIBRATION_SOURCE}")
if not (
    CALIBRATION_SOURCE == "humanevalplus_prompts"
    or CALIBRATION_SOURCE.startswith("hf:")
):
    raise ValueError(
        "CALIBRATION_SOURCE must be 'humanevalplus_prompts' or start with 'hf:'."
    )

## 3) Prune and register model artifacts

**What happens here:** The script `infra/aws/sagemaker/prune_and_register.py` runs **on this machine** (or your notebook instance): it loads the base model, collects activation statistics on calibration text, applies pruning per level, uploads each level’s checkpoint to S3, and writes `model_manifest.json` listing where each level lives.

**Why it can be slow or memory-heavy:** Large models and many calibration samples increase GPU/RAM use and wall-clock time.

**Tips for a first successful run:**

- Keep `MAX_CALIBRATION_SAMPLES` small (e.g. 4–16) and `MAX_CALIBRATION_TOKENS` small (e.g. 256–512).
- Use fewer pruning levels (e.g. `0,20` or just `0`) until the pipeline is verified.
- Ensure Hugging Face access if the model is gated (token / license).

In [ ]:
# -----------------------------------------------------------------------------
# Run local pruning + upload to S3; capture manifest URI from script stdout
# -----------------------------------------------------------------------------
# Environment passed into the subprocess: mirrors SageMakerInfraConfig.from_env() fields
# so the same notebook settings apply to Python scripts that read os.environ.
runtime_env = {
    "AWS_REGION": AWS_REGION,
    "AWS_DEFAULT_REGION": AWS_REGION,
    "SAGEMAKER_ROLE_ARN": SAGEMAKER_ROLE_ARN,
    "PRUNING_ARTIFACT_BUCKET": PRUNING_ARTIFACT_BUCKET,
    "PRUNING_ARTIFACT_PREFIX": PRUNING_ARTIFACT_PREFIX,
    "PRUNING_LOGITS_BUCKET": PRUNING_LOGITS_BUCKET,
    "PRUNING_LOGITS_PREFIX": PRUNING_LOGITS_PREFIX,
    "PRUNING_ENDPOINT_NAME": PRUNING_ENDPOINT_NAME,
    "PRUNING_INSTANCE_TYPE": PRUNING_INSTANCE_TYPE,
    "PRUNING_INSTANCE_COUNT": str(PRUNING_INSTANCE_COUNT),
}

# Build CLI: shlex.quote protects paths and dataset names that contain spaces.
prune_command = " ".join(
    [
        "python",
        "infra/aws/sagemaker/prune_and_register.py",
        f"--base-model-id {shlex.quote(BASE_MODEL_ID)}",
        f"--calibration-source {shlex.quote(CALIBRATION_SOURCE)}",
        f"--pruning-levels {shlex.quote(PRUNING_LEVELS)}",
        f"--max-calibration-samples {MAX_CALIBRATION_SAMPLES}",
        f"--max-calibration-tokens {MAX_CALIBRATION_TOKENS}",
        f"--artifact-bucket {shlex.quote(PRUNING_ARTIFACT_BUCKET)}",
        f"--artifact-prefix {shlex.quote(PRUNING_ARTIFACT_PREFIX)}",
        f"--region {shlex.quote(AWS_REGION)}",
    ]
)

prune_stdout = run_cmd(prune_command, env=runtime_env)
# The script prints a JSON object containing manifest_s3_uri; extract it from mixed logs.
manifest_match = re.search(r"\{[\s\S]*\}", prune_stdout)
if not manifest_match:
    raise RuntimeError("Could not parse manifest JSON from prune script output.")
manifest_payload = json.loads(manifest_match.group(0))
# Used by deploy step: tells the serving container where each pruning level is on S3.
MANIFEST_S3_URI = manifest_payload["manifest_s3_uri"]
print(f"Manifest uploaded to: {MANIFEST_S3_URI}")

## 4) Build and push serving image (AWS CLI + Docker)

**What this section does:** SageMaker runs your model inside a **container image** stored in **Amazon ECR**. This cell **does not** run Docker for you (many hosted notebooks disallow the Docker daemon); it prints the exact commands so you can run them on a machine where Docker and AWS CLI work.

**Typical flow:**

1. Log in to ECR (`get-login-password` piped to `docker login`).
2. Ensure an ECR repository exists (`describe-repositories`, or `create-repository` if missing).
3. `docker build` from `infra/containers/qwen-serving`, **tag** with your account/region/repo, **push**.

If `CONTAINER_IMAGE_URI` already points to a pushed image, you can skip straight to deployment.

In [ ]:
# Compose shell snippets (strings only). Copy-paste into a terminal with Docker running.
# Registry URL pattern: <account>.dkr.ecr.<region>.amazonaws.com

# Authenticate Docker to your private ECR registry for this region.
ecr_login_cmd = (
    f"aws ecr get-login-password --region {shlex.quote(AWS_REGION)} | "
    f"docker login --username AWS --password-stdin "
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com"
)
# Check repository exists (create if your org provisions repos out-of-band and this fails).
create_repo_cmd = (
    "aws ecr describe-repositories "
    f"--repository-names {shlex.quote(ECR_REPOSITORY_NAME)} "
    f"--region {shlex.quote(AWS_REGION)}"
)
create_repo_fallback_cmd = (
    "aws ecr create-repository "
    f"--repository-name {shlex.quote(ECR_REPOSITORY_NAME)} "
    f"--region {shlex.quote(AWS_REGION)}"
)
# Build context: project Dockerfile under infra/containers/qwen-serving
build_cmd = (
    "docker build "
    "-t qwen-serving-local "
    "-f infra/containers/qwen-serving/Dockerfile "
    "infra/containers/qwen-serving"
)
# Tag local image with the ECR URI expected by deploy_endpoint.py
tag_cmd = f"docker tag qwen-serving-local {shlex.quote(CONTAINER_IMAGE_URI)}"
push_cmd = f"docker push {shlex.quote(CONTAINER_IMAGE_URI)}"

print("Run these commands on a machine with Docker if you need to build/push:")
print(ecr_login_cmd)
print(create_repo_cmd)
print("If describe-repositories fails, run:")
print(create_repo_fallback_cmd)
print(build_cmd)
print(tag_cmd)
print(push_cmd)
print(f"Final image URI: {CONTAINER_IMAGE_URI}")

## 5) Deploy or update SageMaker endpoint

**What this section does:** Calls `infra/aws/sagemaker/deploy_endpoint.py`, which:

1. Registers a **SageMaker Model** pointing at your ECR image and environment (manifest URI, logits bucket).
2. Creates a new **Endpoint Configuration** (immutable snapshot of model + instance type/count).
3. Either **creates** the endpoint or **updates** an existing endpoint name to use the new config.

**Dependencies:** `MANIFEST_S3_URI` from the pruning step; `CONTAINER_IMAGE_URI` must exist in ECR; IAM role must allow SageMaker to pull the image and access S3.

The next code cell after deploy **polls** until the endpoint is `InService` (or fails).

In [ ]:
# Guard: deploy requires the manifest produced by prune_and_register (S3 pointer to all levels).
if "MANIFEST_S3_URI" not in globals() or not MANIFEST_S3_URI:
    raise RuntimeError("Run pruning step first so MANIFEST_S3_URI is set.")

# Subprocess matches CLI usage in infra README; container receives MODEL_MANIFEST_S3_URI and logits settings.
deploy_command = " ".join(
    [
        "python",
        "infra/aws/sagemaker/deploy_endpoint.py",
        f"--container-image-uri {shlex.quote(CONTAINER_IMAGE_URI)}",
        f"--manifest-s3-uri {shlex.quote(MANIFEST_S3_URI)}",
        f"--endpoint-name {shlex.quote(PRUNING_ENDPOINT_NAME)}",
        f"--role-arn {shlex.quote(SAGEMAKER_ROLE_ARN)}",
        f"--instance-type {shlex.quote(PRUNING_INSTANCE_TYPE)}",
        f"--instance-count {PRUNING_INSTANCE_COUNT}",
        f"--region {shlex.quote(AWS_REGION)}",
        f"--logits-bucket {shlex.quote(PRUNING_LOGITS_BUCKET)}",
        f"--logits-prefix {shlex.quote(PRUNING_LOGITS_PREFIX)}",
    ]
)
run_cmd(deploy_command, env=runtime_env)

In [ ]:
def wait_for_endpoint_in_service(
    endpoint_name: str,
    region: str,
    timeout_seconds: int = 60 * 30,
    poll_seconds: int = 30,
) -> str:
    """Wait until endpoint enters InService or a terminal failure state.

    SageMaker updates are asynchronous: after deploy returns, instances may still
    start or pull the container image. Polling avoids invoking before ready.
    """

    client = boto3.client("sagemaker", region_name=region)
    started = time.time()
    while True:
        response = client.describe_endpoint(EndpointName=endpoint_name)
        status = response["EndpointStatus"]
        print(f"Endpoint status: {status}")
        if status == "InService":
            return status
        # Terminal failure: surface FailureReason from API for quicker debugging.
        if status in {"Failed", "OutOfService"}:
            raise RuntimeError(
                f"Endpoint entered terminal state: {status}\n"
                f"Failure reason: {response.get('FailureReason', 'unknown')}"
            )
        if time.time() - started > timeout_seconds:
            raise TimeoutError("Timed out waiting for endpoint to become InService.")
        time.sleep(poll_seconds)


wait_for_endpoint_in_service(PRUNING_ENDPOINT_NAME, AWS_REGION)

## 6) Invoke the endpoint with a prompt

**What this section does:** Sends a JSON payload to **SageMaker Runtime** (`invoke_endpoint`). The container’s inference handler expects keys such as `prompt`, `task_id`, `pruning_level`, and `seed` (see `infra/containers/qwen-serving/inference.py`).

**Why boto3 here:** Same API as the CLI would call under the hood; keeping it in the notebook makes it easy to capture `response_payload` for the logits section below.

**Response highlights:**

- `generated_text`: model output for the new tokens only (prompt excluded by the container).
- `logits_s3_uri`: where full-vocab logits for each generated step were stored (large; use for analysis, not always print).

In [ ]:
# Client for InvokeEndpoint (distinct from the SageMaker control-plane client).
runtime_client = boto3.client("sagemaker-runtime", region_name=AWS_REGION)

# Example user prompt and metadata (task_id is echoed back; useful for tracing logs).
SAMPLE_PROMPT = (
    "Write a Python function `is_palindrome(s: str) -> bool` "
    "that ignores casing and non-alphanumeric characters."
)
REQUEST_TASK_ID = "demo-notebook-task-001"
# Use first configured pruning level so we know that checkpoint exists in the manifest.
REQUEST_PRUNING_LEVEL = int(PRUNING_LEVELS.split(",")[0])
REQUEST_SEED = 123

# Body schema matches invoke_endpoint.py and the serving container’s input_fn.
request_payload = {
    "prompt": SAMPLE_PROMPT,
    "task_id": REQUEST_TASK_ID,
    "pruning_level": REQUEST_PRUNING_LEVEL,
    "seed": REQUEST_SEED,
    "max_new_tokens": 128,
    "temperature": 0.0,
    "top_p": 1.0,
}

response = runtime_client.invoke_endpoint(
    EndpointName=PRUNING_ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(request_payload).encode("utf-8"),
)
# streaming body -> bytes -> JSON dict
response_payload = json.loads(response["Body"].read().decode("utf-8"))

print("Response metadata (identifiers and logits location):")
print(
    json.dumps(
        {
            "task_id": response_payload.get("task_id"),
            "pruning_level": response_payload.get("pruning_level"),
            "token_count": response_payload.get("token_count"),
            "request_id": response_payload.get("request_id"),
            "logits_s3_uri": response_payload.get("logits_s3_uri"),
        },
        indent=2,
    )
)
print("\nGenerated text:\n")
print(response_payload.get("generated_text", ""))

## 7) Compute token log probabilities from the logits artifact

**What the endpoint returns:** The model text in `generated_text`, plus `logits_s3_uri` pointing at an object in S3. The reference container writes **one JSON line per generated token**, each line containing the **full vocabulary logits** for that step (large).

**What we compute here:**

1. Download the artifact from S3.
2. For each step (row), apply **log-softmax** across the vocabulary → log probability distribution.
3. **Index** that distribution at the **token id that was actually generated** → `selected_logprob` for that token.

**Optional `.npz`:** If you later save logits as NumPy archives, `load_logits_artifact` can load `generated_token_ids` and `logits` arrays the same way.

**Accessibility note:** Full vocab sizes can be huge; this notebook only **summarizes** and **prints a short preview** so outputs stay readable.

In [ ]:
def load_logits_artifact(s3_uri: str, region: str) -> tuple[np.ndarray, np.ndarray]:
    """Download and parse logits artifact from S3.

    The serving container writes JSONL (one object per line). Each line includes
    `token_id` and `logits` (full vocabulary). For `.npz`, expects arrays
    `generated_token_ids` and `logits` with one row per generated step.

    Parameters
    ----------
    s3_uri:
        S3 URI from inference response metadata.
    region:
        AWS region used for S3 client.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Tuple of (generated_token_ids, logits_2d_array).
    """

    bucket, key = parse_s3_uri(s3_uri)
    s3 = boto3.client("s3", region_name=region)
    # Read entire object into memory; for very long outputs consider streaming.
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    if key.endswith(".npz"):
        with np.load(io.BytesIO(body)) as npz_data:
            token_ids = np.array(npz_data["generated_token_ids"], dtype=np.int64)
            logits = np.array(npz_data["logits"], dtype=np.float32)
        return token_ids, logits

    # JSONL: stack rows into shape (num_new_tokens, vocab_size).
    lines = body.decode("utf-8").splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    token_ids = np.array([int(row["token_id"]) for row in rows], dtype=np.int64)
    logits = np.array([row["logits"] for row in rows], dtype=np.float32)
    return token_ids, logits


# Block: pull logits location from the invoke response (same request_id ties logs together).
logits_s3_uri = response_payload.get("logits_s3_uri")
if not logits_s3_uri:
    raise RuntimeError("No logits_s3_uri in endpoint response.")

generated_token_ids, step_logits = load_logits_artifact(logits_s3_uri, AWS_REGION)
if generated_token_ids.size == 0 or step_logits.size == 0:
    raise RuntimeError("Logits artifact is empty; cannot compute log probabilities.")

# Per-step full distribution in log space; select probability mass at chosen token id.
step_logprobs = log_softmax(step_logits)
selected_logprobs = step_logprobs[
    np.arange(generated_token_ids.shape[0]), generated_token_ids
]

print(f"Loaded {generated_token_ids.shape[0]} generated tokens.")
print(
    "Average selected-token log probability: "
    f"{float(np.mean(selected_logprobs)):.4f}"
)
print(
    "Average selected-token probability: "
    f"{float(np.mean(np.exp(selected_logprobs))):.4f}"
)

In [ ]:
# Decode token ids with the same tokenizer family as the base model for human-readable strings.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
preview_count = min(25, generated_token_ids.shape[0])

print("First generated tokens with selected-token log probabilities:")
for index in range(preview_count):
    token_id = int(generated_token_ids[index])
    token_text = tokenizer.decode([token_id])
    token_logprob = float(selected_logprobs[index])
    token_prob = float(np.exp(token_logprob))
    print(
        f"idx={index:03d} token_id={token_id:>8} "
        f"logp={token_logprob:>8.4f} p={token_prob:>8.6f} "
        f"token={token_text!r}"
    )

In [ ]:
# -----------------------------------------------------------------------------
# Optional: same invoke as a subprocess CLI (matches infra/aws/sagemaker/invoke_endpoint.py)
# -----------------------------------------------------------------------------
# Uncomment run_cmd(...) below if you want identical behavior to terminal usage.
invoke_script_cmd = " ".join(
    [
        "python",
        "infra/aws/sagemaker/invoke_endpoint.py",
        f"--endpoint-name {shlex.quote(PRUNING_ENDPOINT_NAME)}",
        f"--prompt {shlex.quote(SAMPLE_PROMPT)}",
        f"--task-id {shlex.quote(REQUEST_TASK_ID)}",
        f"--pruning-level {REQUEST_PRUNING_LEVEL}",
        f"--seed {REQUEST_SEED}",
        "--max-new-tokens 128",
        "--temperature 0.0",
        "--top-p 1.0",
        f"--region {shlex.quote(AWS_REGION)}",
    ]
)
print(invoke_script_cmd)
# run_cmd(invoke_script_cmd, env=runtime_env)

## 8) Cleanup and cost controls

**Billing reminder:** A SageMaker endpoint keeps underlying instances allocated while it exists. Delete the endpoint when you no longer need low-latency inference.

**Suggested teardown order:**

1. Delete the **endpoint** (stops compute charges from that endpoint).
2. Delete unused **endpoint configurations** (named snapshots).
3. Delete unused **models** (metadata pointing at image + optional model data).
4. Optionally remove **S3** prefixes for checkpoints or logits if you no longer need them.

The cleanup cell below is **off by default** (`RUN_CLEANUP = False`) so you do not accidentally delete production resources.

In [ ]:
# SageMaker control plane: delete endpoint/config/model (not runtime invoke).
sm_client = boto3.client("sagemaker", region_name=AWS_REGION)

# Safety gate: flip to True only when you intend to delete AWS resources.
RUN_CLEANUP = False

if RUN_CLEANUP:
    print(f"Deleting endpoint: {PRUNING_ENDPOINT_NAME}")
    sm_client.delete_endpoint(EndpointName=PRUNING_ENDPOINT_NAME)

    # deploy_endpoint.py timestamps config/model names; list_* finds related objects.
    # Note: deleting ALL matching names may remove stacks from other experiments—review names first.
    endpoint_configs = sm_client.list_endpoint_configs(
        NameContains="qwen-pruning-config"
    ).get("EndpointConfigs", [])
    for config in endpoint_configs:
        config_name = config["EndpointConfigName"]
        print(f"Deleting endpoint config: {config_name}")
        sm_client.delete_endpoint_config(EndpointConfigName=config_name)

    models = sm_client.list_models(NameContains="qwen-pruning-model").get("Models", [])
    for model in models:
        model_name = model["ModelName"]
        print(f"Deleting model: {model_name}")
        sm_client.delete_model(ModelName=model_name)

    print("Cleanup requests submitted.")
else:
    print(
        "Cleanup is disabled. Set RUN_CLEANUP=True to delete endpoint/config/model resources."
    )

In [ ]:
# These are destructive S3 operations—run only if you accept permanent data loss for those prefixes.
print("Artifact cleanup command templates (review bucket/prefix before running):")
print(
    "aws s3 rm "
    f"s3://{PRUNING_ARTIFACT_BUCKET}/{PRUNING_ARTIFACT_PREFIX} "
    "--recursive"
)
print(
    "aws s3 rm "
    f"s3://{PRUNING_LOGITS_BUCKET}/{PRUNING_LOGITS_PREFIX} "
    "--recursive"
)